# Pipeline Test
If you intend to run the code in this repository, make sure this notebook runs on your computer as a sanity check. 
Different pitfalls have been tested here (e. g. wrong installation of NumPy version, issues with MNE library, ...).

In [ ]:
# Addition for conflicting math libraries on Mac

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

print("Environment flags set, Kernel should now be stable")
os.chdir('/Users/anton/Documents/CNN-Transformer-EEG-BCI-26')

Environment flags set, Kernel should now be stable


In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import scipy
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
%matplotlib inline

from preprocess import load_subject_data
from dataset import PreprocessedDataset
from datamodule import create_dataloaders
from model import EEGClassifier, BaselineCNN
from train import train, plot_metrics
from evaluate import evaluate, get_predictions, visualize_predictions

print(f"Numpy: {np.__version__}") # Should be 1.26.x
print(f"SciPy: {scipy.__version__}") # Should be 1.11+
print("Kernel started successfully!")

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

try:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
except NameError:
    PROJECT_ROOT = os.getcwd()

print(f"Project Root identified as: {PROJECT_ROOT}")
processed_dir = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

Project Root identified as: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26
Project Root identified as: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26
Project Root identified as: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26
Project Root identified as: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26
Numpy: 1.26.4
SciPy: 1.12.0
Kernel started successfully!
Using device: mps
Project Root identified as: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/src


## 1. Testing Preprocessing

In [3]:
print("Testing the preprocessing")
test_subjects = [1, 2]
X_list, y_list = [], []

for s in test_subjects:
    print(f"Processing subject {s}...")
    X, y = load_subject_data(subject_id=s)
    X_list.append(X)
    y_list.append(y)

X_final = np.concatenate(X_list, axis=0)
y_final = np.concatenate(y_list, axis=0)

os.makedirs(processed_dir, exist_ok=True)
np.save(os.path.join(processed_dir, 'eeg_X_processed.npy'), X_final)
np.save(os.path.join(processed_dir, 'eeg_y_processed.npy'), y_final)

print(f"Success! X shape: {X_final.shape}, y shape: {y_final.shape}")

Testing the preprocessing
Processing subject 1...
Checkpoint 1: Loading EDF files
Extracting EDF parameters from /Users/anton/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /Users/anton/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /Users/anton/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Checkpoint 2: Annotations and Montage
Used Annotations descriptions: ['T0', 'T1', 'T2']
Checkpoint 3: Filtering
Filtering raw data in 3 contiguous segments
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
--

In [4]:
print("Testing the dataloaders")

# Use a tiny batch size for testing
batch_size = 2
try:
    train_loader, val_loader, test_loader = create_dataloaders(batch_size=batch_size)
    print("Dataloaders created successfully!")

    # Grab one batch from the train_loader
    batch_X, batch_y = next(iter(train_loader))

    print(f"Batch X shape: {batch_X.shape}") # Expected: (batch_size, 64, samples)
    print(f"Batch y shape: {batch_y.shape}") # Expected: (batch_size,)
    print(f"Batch X dtype: {batch_X.dtype}") # Expected: torch.float32
    print(f"Batch y dtype: {batch_y.dtype}") # Expected: torch.float32

    # Simple check: does it have values?
    print(f"X mean: {batch_X.mean():.4f}")

except Exception as e:
    print(f"Dataloader Error: {e}")

Testing the dataloaders
Subjects, train: 61, val: 9, test: 18
Dataloaders created successfully!
Batch X shape: torch.Size([2, 64, 481])
Batch y shape: torch.Size([2])
Batch X dtype: torch.float32
Batch y dtype: torch.float32
X mean: 0.1023


### Shape Explanations
Subjects performed each approximately 15 trials for a task and in this case the experimental runs 4, 8 and 12 were selected -> 2 (subs) x 3 (runs) x 15 (trials) = 90; here we get 88 which could be because the second subject just performed 13 trials.
With a 70/10/20 split that gives 61 training recordings, 9 for validation and 18 for testing.

#### Early Diagnostics 
The first output of shapes can already give a hint of whether data was imported wrongly. By knowing what each number means, diagnosing maleable behavior can be detected early on. In this case no unusual behavior has been detected:

* Batch X shape: 2 numbers of trials in batch, 64 channels/electrodes and 481 samples/time-points in each trial
* Samples: The individual time-points (bins) recorded at the sampling rate of 160 Hz (calculated: 3s x 160 Hz = 1 = 480 samples)
* X mean: should be close to 0 to show that the z-score normalization is working as it should :)

## 2. Testing the Main Architecture
### 2.1 Overfitting Artificial Data
To check the learning process small input data matching the original data shape is created and then passed onto the main CNN-Classifier (using a reduced embedding dimension and fewer heads for the MHA to save computation ressources), which is trained for 10 epochs to verify whether the data pipeline fundamentally works.

In [5]:
# Dummy data - 10 trials, 64 channels, 481 samples
X_tiny = np.random.randn(10, 64, 481).astype(np.float32)
y_tiny = np.random.randint(0, 2, 10).astype(np.float32)

np.save(os.path.join(processed_dir, 'eeg_X_processed.npy'), X_final)
np.save(os.path.join(processed_dir, 'eeg_y_processed.npy'), y_final)

In [6]:
batch_size = 2
train_loader, val_loader, test_loader = create_dataloaders(batch_size=batch_size)

model_test_main = EEGClassifier(
    n_channels=64, 
    emb_dim=32, 
    n_heads=2, 
    dropout=0.1, 
    fs=160
)

optimizer_test_main = optim.Adam(model_test_main.parameters(), lr=0.001, weight_decay=1e-4)

Subjects, train: 61, val: 9, test: 18


In [7]:
n_epochs_test = 10

trained_model, history = train(
    model=model_test_main,
    n_epochs=n_epochs_test,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_test_main,
    device=device
)

print('Dry run completed with success.')
plot_metrics(history)

Epoch 1/10 | Train Loss 0.7290 | Val Kappa: 0.0000
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 2/10 | Train Loss 0.7295 | Val Kappa: 0.3077
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 3/10 | Train Loss 0.6882 | Val Kappa: 0.0000
Epoch 4/10 | Train Loss 0.7030 | Val Kappa: 0.0000
Epoch 5/10 | Train Loss 0.6711 | Val Kappa: -0.2273
Epoch 6/10 | Train Loss 0.6686 | Val Kappa: 0.7805
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 7/10 | Train Loss 0.4379 | Val Kappa: 0.7805
Epoch 8/10 | Train Loss 0.2335 | Val Kappa: 1.0000
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 9/10 | Train Loss 0.1793 | Val Kappa: 0.7805
Epoch 10/10 | Train Loss 0.0866 | Val Kappa: 0.7805
Dry run complete

/Users/anton/Documents/CNN-Transformer-EEG-BCI-26/src/train.py:166: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Test Baseline Model
model_baseline_test = BaselineCNN()
optimizer_baseline_test = optim.Adam(model_baseline_test.parameters(), lr=0.001, weight_decay=1e-4)

trained_baseline_test, history_baseline = train(
    model=model_baseline_test,
    n_epochs=n_epochs_test,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_baseline_test,
    device=device
)

plot_metrics(history_baseline)

Epoch 1/10 | Train Loss 0.7181 | Val Kappa: 0.0000
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 2/10 | Train Loss 0.6824 | Val Kappa: 0.3415
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 3/10 | Train Loss 0.6363 | Val Kappa: 0.1429
Epoch 4/10 | Train Loss 0.6067 | Val Kappa: 0.3415
Epoch 5/10 | Train Loss 0.5355 | Val Kappa: -0.2857
Epoch 6/10 | Train Loss 0.4606 | Val Kappa: 0.3415
Epoch 7/10 | Train Loss 0.4606 | Val Kappa: -0.0465
Epoch 8/10 | Train Loss 0.4643 | Val Kappa: 0.3415
Epoch 9/10 | Train Loss 0.4969 | Val Kappa: 0.3415
Epoch 10/10 | Train Loss 0.3545 | Val Kappa: 0.3415


/Users/anton/Documents/CNN-Transformer-EEG-BCI-26/src/train.py:166: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.2 Testing Models on Small Original Data

In [9]:
print("Preprocessing subjects 1 and 2...")
X1, y1 = load_subject_data(subject_id=1)
X2, y2 = load_subject_data(subject_id=2)

X = np.concatenate([X1, X2], axis=0)
y = np.concatenate([y1, y2], axis=0)

dataset = PreprocessedDataset(X, y)


np.save(os.path.join(processed_dir, 'eeg_X_processed.npy'), X)
np.save(os.path.join(processed_dir, 'eeg_y_processed.npy'), y)

print(f"Success! Tiny dataset created. X shape: {X.shape}")

Preprocessing subjects 1 and 2...
Checkpoint 1: Loading EDF files
Extracting EDF parameters from /Users/anton/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /Users/anton/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /Users/anton/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Checkpoint 2: Annotations and Montage
Used Annotations descriptions: ['T0', 'T1', 'T2']
Checkpoint 3: Filtering
Filtering raw data in 3 contiguous segments
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
------------------

In [10]:
batch_size = 2
train_loader, val_loader, test_loader = create_dataloaders(batch_size=batch_size)

model_test_main = EEGClassifier(
    n_channels=64, 
    emb_dim=32, 
    n_heads=2, 
    dropout=0.1, 
    fs=160
)

optimizer_test_main = optim.Adam(model_test_main.parameters(), lr=0.001, weight_decay=1e-4)

Subjects, train: 61, val: 9, test: 18


In [11]:
n_epochs_test = 20

trained_model, history = train(
    model=model_test_main,
    n_epochs=n_epochs_test,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_test_main,
    device=device
)

print('Dry run completed with success.')
plot_metrics(history)

Epoch 1/20 | Train Loss 0.6710 | Val Kappa: 0.0000
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 2/20 | Train Loss 0.7184 | Val Kappa: 0.0000
Epoch 3/20 | Train Loss 0.7106 | Val Kappa: 0.0000
Epoch 4/20 | Train Loss 0.6939 | Val Kappa: 0.3721
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 5/20 | Train Loss 0.6218 | Val Kappa: 0.7805
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 6/20 | Train Loss 0.3999 | Val Kappa: 1.0000
Saved best model to: /Users/anton/Documents/CNN-Transformer-EEG-BCI-26/results/models/best_model_name_lr0.001_bs2.pth
Epoch 7/20 | Train Loss 0.3206 | Val Kappa: 0.7805
Epoch 8/20 | Train Loss 0.1285 | Val Kappa: 0.7805
Epoch 9/20 | Train Loss 0.1184 | Val Kappa: 1.0000
Epoch 10/20 | Train Loss 0.0276 | Val Kappa: 0.5714
Epoch 11/20 | Tra

/Users/anton/Documents/CNN-Transformer-EEG-BCI-26/src/train.py:166: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
criterion = nn.BCEWithLogitsLoss()

evaluate(trained_model, test_loader, criterion, device)
y_true, y_pred = get_predictions(trained_model, test_loader, device)
visualize_predictions(y_true, y_pred)

Prediction Distribution:
{0: 9, 1: 9}
Cohen's Kappa: 0.7778


/Users/anton/Documents/CNN-Transformer-EEG-BCI-26/src/evaluate.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
